# Example Usage of Metics in TopicGPT: 20 Newsgroups Dataset

In this notebook, we will use the 20 Newsgroups dataset to demonstrate the use of the topicgpt package

In [1]:
import sys
import os

# Add the 'src' directory to the Python path
sys.path.append(os.path.abspath("../src"))

### Configurations

TRAIN = True
PROVIDER = "openai"  # "openai" or "anthropic" or "gemini"
EMBEDDING_PATH = f"./SavedEmbeddings/{PROVIDER}_news.pkl"

In [2]:
# select your own API key here. (Note: This specific code will not work for you unless you specified an environment variable for OPENAI_API_KEY)
import os
from dotenv import load_dotenv

load_dotenv()

if PROVIDER == "openai":
    api_key = os.environ.get('OPENAI_API_KEY')
    prompting_model = "gpt-3.5-turbo"
    embedding_model = "text-embedding-3-small"
elif PROVIDER == "gemini":
    api_key = os.environ.get('GEMINI_API_KEY')
    prompting_model = "gemini-2.0-flash-lite"
    embedding_model = "gemini-embedding-001"
elif PROVIDER == "anthropic":
    api_key = os.environ.get('ANTHROPIC_API_KEY')
    prompting_model = "claude-sonnet-4-20250514"
    embedding_model = "all-MiniLM-L6-v2"

### Load Data

In [3]:
from sklearn.datasets import fetch_20newsgroups

data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes')) #download the 20 Newsgroups dataset
corpus = data['data']
corpus = [doc for doc in corpus if doc != ""]

## Initialize and fit the model 

In [ ]:
from topicgpt.TopicGPT import TopicGPT
if TRAIN:
    tm = TopicGPT(
        prompting_model=prompting_model,
        api_key=api_key,
        n_topics=20,  # select 20 topics since the true number of topics is 20
        embedding_model=embedding_model, 
        use_saved_embeddings=False,  # set to False to train the model from scratch
    )
    tm.fit(corpus)  # train the model on the corpus
    tm.save_embeddings(EMBEDDING_PATH) #save the embeddings for future use
else:
    tm = TopicGPT(
        api_key=api_key,
        n_topics=20,  # select 20 topics since the true number of topics is 20
        path_saved_embeddings=EMBEDDING_PATH,
        use_saved_embeddings=True,  # set to True to use saved embeddings
    )

d:\Research\GPTopic\topicgpt-venv\Lib\site-packages\numba\np\ufunc\dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
d:\Research\GPTopic\topicgpt-venv\Lib\site-packages\numba\np\ufunc\dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
d:\Research\GPTopic\topicgpt-venv\Lib\site-packages\numba\np\ufunc\dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaW

Removed 0 empty documents.
Computing vocabulary...


Processing corpus: 100%|██████████| 18466/18466 [00:17<00:00, 1051.77it/s]


Most common words in the vocabulary:
n't: 17116
would: 10872
one: 10144
people: 6442
like: 6409
get: 5815
know: 5742
also: 5588
use: 4987
think: 4982
Computing embeddings...


 11%|█         | 1957/18466 [15:57<1:59:26,  2.30it/s] 

In [ ]:
tm

TopicGPT object with the following parameters:
------------------------------------------------------------------------------------------------------------------------------------------------------
n_topics: 20
prompting_model: gpt-3.5-turbo
max_number_of_tokens: 16384
corpus_instruction: 
embedding_model: text-embedding-3-small
clusterer: <topicgpt.Clustering.Clustering_and_DimRed object at 0x000001B7EE732060>
n_topwords: 2000
n_topwords_description: 500
topword_extraction_methods: ['tfidf', 'cosine_similarity']
compute_vocab_hyperparams: {'verbose': True}
enhancer: TopwordEnhancement(model = gpt-3.5-turbo)
topic_prompting: <topicgpt.TopicPrompting.TopicPrompting object at 0x000001B7EE45E180>

## Get an overview over the identified topics

In [ ]:
### Some information about the trained model
print(f'Number of documents: {len(corpus)}')
print(f'document embedding shape: {tm.document_embeddings.shape}')
print(f'number of vocab: {len(list(tm.vocab_embeddings.keys()))}')
print(f'vocab embedding shape: {list(tm.vocab_embeddings.values())[0].shape}') #shape of a single vocab embedding

Number of documents: 18466
document embedding shape: (18466, 1536)
number of vocab: 21143
vocab embedding shape: (1536,)


In [ ]:
# We need the list of topics to compute the metrics which is done extract_topics()
tm.extract_topics(corpus)

UMAP(angular_rp_forest=True, metric='cosine', min_dist=0, n_components=5, n_jobs=1, random_state=42, verbose=True)
Wed Sep 17 04:11:20 2025 Construct fuzzy simplicial set
Wed Sep 17 04:11:20 2025 Finding Nearest Neighbors
Wed Sep 17 04:11:20 2025 Building RP forest with 12 trees


d:\Research\GPTopic\topicgpt-venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Wed Sep 17 04:11:21 2025 NN descent for 14 iterations
	 1  /  14
	 2  /  14
	 3  /  14
	 4  /  14
	 5  /  14
	 6  /  14
	Stopping threshold met -- exiting after 6 iterations
Wed Sep 17 04:11:23 2025 Finished Nearest Neighbor Search
Wed Sep 17 04:11:23 2025 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Wed Sep 17 04:11:29 2025 Finished embedding


...

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs


Computing word-topic matrix: 100%|██████████| 20/20 [00:16<00:00,  1.22it/s]


Epochs completed:   0%|            0/30 [00:00]

	completed  0  /  30 epochs
	completed  3  /  30 epochs
	completed  6  /  30 epochs
	completed  9  /  30 epochs
	completed  12  /  30 epochs
	completed  15  /  30 epochs
	completed  18  /  30 epochs
	completed  21  /  30 epochs
	completed  24  /  30 epochs
	completed  27  /  30 epochs


[Topic: 0,
 Topic: 1,
 Topic: 2,
 Topic: 3,
 Topic: 4,
 Topic: 5,
 Topic: 6,
 Topic: 7,
 Topic: 8,
 Topic: 9,
 Topic: 10,
 Topic: 11,
 Topic: 12,
 Topic: 13,
 Topic: 14,
 Topic: 15,
 Topic: 16,
 Topic: 17,
 Topic: 18]

In [ ]:
tm.score()

Using topic_lis as source (19 topics)
Generating descriptions for topics using tm.describe_topics...


100%|██████████| 19/19 [00:15<00:00,  1.26it/s]


{'Average_Document_Similarity': 0.29395020352298806,
 'Average_Document_Cohesion': 0.30329}